# LeetCode 3430 — Maximum and Minimum Sums of at Most Size K Subarrays bằng Python

- Link: https://leetcode.com/problems/maximum-and-minimum-sums-of-at-most-size-k-subarrays/
- Độ khó: **Hard**
- Chủ đề: **Kỹ thuật đóng góp** (mỗi phần tử là max/min của bao nhiêu mảng con), **ngăn xếp đơn điệu**, **đếm cặp (x, y) với x + y ≤ k + 1** bằng công thức đóng

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def minMaxSubarraySum(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Đổi "tổng của max (min) trên mọi mảng con" thành "**mỗi phần tử** đóng góp `giá trị × số mảng con mà nó là max (min)`".
2. Dùng **ngăn xếp đơn điệu** để tìm phạm vi trái/phải mà phần tử còn là max, với quy tắc **phá hòa** (một bên `<`, một bên `≤`) để không đếm trùng.
3. Thêm ràng buộc độ dài `≤ k`: đếm cặp `(x, y)` (số phần tử lấy về bên trái/phải, tính cả chính nó) với `1 ≤ x ≤ a`, `1 ≤ y ≤ b`, `x + y − 1 ≤ k`.
4. Rút ra công thức đóng `O(1)` cho số cặp — hai cách: tách khúc và **bao hàm – loại trừ**.
5. Xử lý min bằng cách đổi dấu mảng (dùng lại code max).

## 2. Đề bài & ví dụ

Cho `nums` và `k`. Trả về tổng `(max + min)` của **mọi** mảng con có **tối đa `k`** phần tử.

**Ví dụ 1:** `nums = [1,2,3], k = 2` → `20`. **Ví dụ 2:** `nums = [1,-3,1], k = 2` → `-6`.

**Ràng buộc:** `1 ≤ n ≤ 8·10^4`, `1 ≤ k ≤ n`, `|nums[i]| ≤ 10^6`.

In [1]:
from typing import List

examples = [([1, 2, 3], 2, 20), ([1, -3, 1], 2, -6)]
for nums, k, exp in examples:
    print(nums, k, '-> kỳ vọng', exp)

[1, 2, 3] 2 -> kỳ vọng 20
[1, -3, 1] 2 -> kỳ vọng -6


## 3. Kiến thức nền

### 3.1. Ngăn xếp đơn điệu: phần tử lớn hơn gần nhất

Duyệt trái → phải, giữ ngăn xếp chỉ số với giá trị **giảm dần**; khi gặp `nums[i]`, bật mọi chỉ số có giá trị nhỏ hơn. Đỉnh còn lại là "phần tử ≥ gần nhất bên trái".

In [2]:
def prev_ge(arr):
    res, st = [], []
    for i, x in enumerate(arr):
        while st and arr[st[-1]] < x:
            st.pop()
        res.append(st[-1] if st else -1)
        st.append(i)
    return res

print(prev_ge([3, 1, 2, 2, 5]))

[-1, 0, 0, 2, -1]


### 3.2. Đóng góp và phá hòa

Phần tử `i` là max của mảng con `[l, r]` (`l ≤ i ≤ r`) nếu mọi phần tử trong đó ≤ `nums[i]`. Khi có giá trị **bằng nhau**, quy ước max là phần tử **trái nhất**: bên trái dừng ở phần tử `≥` (không vượt qua phần tử bằng), bên phải dừng ở phần tử `>`… — hoặc ngược lại, miễn nhất quán. Khi đó mỗi mảng con được gán cho đúng một phần tử.

Gọi `a = i − p` (số lựa chọn đầu trái: `l ∈ (p, i]`), `b = q − i` (số lựa chọn đầu phải). Không giới hạn độ dài thì số mảng con là `a · b`.

### 3.3. Đếm cặp với ràng buộc độ dài

Viết `x = i − l + 1 ∈ [1, a]`, `y = r − i + 1 ∈ [1, b]`; độ dài `x + y − 1 ≤ k` ⇔ `x + y ≤ k + 1`.

In [3]:
def count_pairs_brute(a, b, k):
    return sum(1 for x in range(1, a + 1) for y in range(1, b + 1) if x + y <= k + 1)

print(count_pairs_brute(3, 4, 100), '= 3·4;', count_pairs_brute(3, 4, 3), count_pairs_brute(5, 5, 4))

12 = 3·4; 6 10


## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Ví dụ 1 `[1, 2, 3]`, `k = 2`, phần **max**:

- `3` (i = 2): `a = 3` (không có phần tử ≥ bên trái), `b = 1`. Cặp `x + y ≤ 3` với `y = 1`: `x ∈ {1, 2}` ⇒ 2 mảng con (`[3]`, `[2,3]`) ⇒ đóng góp `6`.
- `2` (i = 1): `a = 2`, `b = 1` (bị `3` chặn) ⇒ `x ∈ {1, 2}` ⇒ 2 mảng con ⇒ `4`.
- `1`: `a = 1`, `b = 1` ⇒ 1 ⇒ `1`.

Tổng max `= 11`. Phần min tương tự cho `9` ⇒ `20` ✓.

## 5. Cách 1 — Brute Force: mọi mảng con độ dài ≤ k

In [4]:
def minmax_brute(nums, k):
    n, total = len(nums), 0
    for i in range(n):
        mn = mx = nums[i]
        for j in range(i, min(n, i + k)):
            mn = min(mn, nums[j]); mx = max(mx, nums[j])
            total += mn + mx
    return total

for nums, k, exp in examples:
    print(minmax_brute(nums, k), '| kỳ vọng', exp)

20 | kỳ vọng 20
-6 | kỳ vọng -6


**Độ phức tạp:** `O(n · k)` — tới `6.4·10^9` khi `k ≈ n`.

## 6. Ý tưởng thuật toán / bước đột phá

`Tổng = Σ_i nums[i] · C_max(i) + Σ_i nums[i] · C_min(i)` với `C(i) = cnt(a_i, b_i, k)`.

**Công thức đóng cho `cnt(a, b, k)`:** `x` chỉ chạy tới `a' = min(a, k)`. Với `x ≤ k + 1 − b`, `y` lấy đủ `b` giá trị; với `x` lớn hơn, `y` lấy `k + 1 − x` giá trị. Đặt `c = clamp(k + 1 − b, 0, a')`:

`cnt = c · b + Σ_{x=c+1}^{a'} (k + 1 − x) = c · b + (a' − c)(k + 1) − (a'(a'+1)/2 − c(c+1)/2)`.

Phần min: chạy lại với mảng đổi dấu và đổi dấu kết quả.

In [5]:
def cnt_closed(a, b, k):
    a2 = min(a, k)
    c = max(0, min(a2, k + 1 - b))
    return c * b + (a2 - c) * (k + 1) - (a2 * (a2 + 1) // 2 - c * (c + 1) // 2)

assert all(cnt_closed(a, b, k) == count_pairs_brute(a, b, k) for a in range(1, 9) for b in range(1, 9) for k in range(1, 18))
print('Công thức đóng khớp đếm trực tiếp với a, b ≤ 8, k ≤ 17.')

Công thức đóng khớp đếm trực tiếp với a, b ≤ 8, k ≤ 17.


## 7. Các cách tiếp theo

### 7.1. Cách 2 — Đóng góp + đếm cặp bằng vòng lặp `O(min(a, k))`

In [6]:
def extents(arr):
    n = len(arr)
    left, st = [0] * n, []
    for i in range(n):
        while st and arr[st[-1]] < arr[i]:
            st.pop()
        left[i] = i - (st[-1] if st else -1)
        st.append(i)
    right, st = [0] * n, []
    for i in range(n - 1, -1, -1):
        while st and arr[st[-1]] <= arr[i]:
            st.pop()
        right[i] = (st[-1] if st else n) - i
        st.append(i)
    return left, right

def cnt_loop(a, b, k):
    return sum(min(b, k + 1 - x) for x in range(1, min(a, k) + 1))

def minmax_contrib(nums, k, cnt):
    total = 0
    for sign in (1, -1):
        arr = [sign * x for x in nums]
        left, right = extents(arr)
        total += sign * sum(arr[i] * cnt(left[i], right[i], k) for i in range(len(arr)))
    return total

for nums, k, exp in examples:
    print(minmax_contrib(nums, k, cnt_loop), '| kỳ vọng', exp)

20 | kỳ vọng 20
-6 | kỳ vọng -6


Đúng nhưng mỗi phần tử tốn `O(min(a, k))` ⇒ tệ nhất `O(n · k)` (mảng tăng dần).

### 7.2. Cách 3 — Đóng góp + công thức đóng (lời giải chính)

In [7]:
def minmax_closed(nums, k):
    return minmax_contrib(nums, k, cnt_closed)

for nums, k, exp in examples:
    print(minmax_closed(nums, k), '| kỳ vọng', exp)

20 | kỳ vọng 20
-6 | kỳ vọng -6


### 7.3. Cách 4 — Công thức bằng bao hàm – loại trừ

`h(s)` = số cặp `x, y ≥ 1` với `x + y ≤ s` `= s(s−1)/2` (với `s ≥ 1`). Bỏ các cặp `x > a` (đặt `x' = x − a`) và `y > b`, cộng lại phần trừ hai lần:

`cnt(a, b, k) = h(k+1) − h(k+1−a) − h(k+1−b) + h(k+1−a−b)` (với `h(s) = 0` khi `s < 1`).

In [8]:
def h(s):
    return s * (s - 1) // 2 if s >= 1 else 0

def cnt_ie(a, b, k):
    s = k + 1
    return h(s) - h(s - a) - h(s - b) + h(s - a - b)

assert all(cnt_ie(a, b, k) == count_pairs_brute(a, b, k) for a in range(1, 9) for b in range(1, 9) for k in range(1, 18))
for nums, k, exp in examples:
    print(minmax_contrib(nums, k, cnt_ie), '| kỳ vọng', exp)

20 | kỳ vọng 20
-6 | kỳ vọng -6


## 8. TRACE từng bước

Trace cách 3 trên ví dụ 2 (`[1, -3, 1]`, `k = 2`): in phạm vi `a, b`, số mảng con và đóng góp cho cả max và min.

In [9]:
def trace(nums, k):
    total = 0
    for sign, name in ((1, 'max'), (-1, 'min')):
        arr = [sign * x for x in nums]
        left, right = extents(arr)
        for i in range(len(nums)):
            c = cnt_closed(left[i], right[i], k)
            total += nums[i] * c
            print(f"{name}: i={i} giá trị {nums[i]:>3}, a={left[i]}, b={right[i]} -> {c} mảng con, đóng góp {nums[i] * c}")
    print('tổng:', total)

trace([1, -3, 1], 2)

max: i=0 giá trị   1, a=1, b=3 -> 2 mảng con, đóng góp 2
max: i=1 giá trị  -3, a=1, b=1 -> 1 mảng con, đóng góp -3
max: i=2 giá trị   1, a=2, b=1 -> 2 mảng con, đóng góp 2
min: i=0 giá trị   1, a=1, b=1 -> 1 mảng con, đóng góp 1
min: i=1 giá trị  -3, a=2, b=2 -> 3 mảng con, đóng góp -9
min: i=2 giá trị   1, a=1, b=1 -> 1 mảng con, đóng góp 1
tổng: -6


Chú ý hai số `1` ở phần max: số `1` bên **trái** có `a = 1`, `b = 3` (bên phải không có phần tử **lớn hơn hẳn** nên nó "sở hữu" cả `[1, -3, 1]`); số `1` bên **phải** có `a = 2` (dừng ở số `1` trái vì điều kiện `≥`), `b = 1`. Nhờ vậy mảng con `[1, -3, 1]` chỉ được gán cho một phần tử. Với `k = 2` công thức chỉ đếm các mảng con độ dài ≤ 2 trong phạm vi đó.

## 9. Code nộp LeetCode

In [10]:
class Solution:
    def minMaxSubarraySum(self, nums: List[int], k: int) -> int:
        n = len(nums)

        def count(a, b):
            a2 = a if a < k else k
            c = k + 1 - b
            if c < 0:
                c = 0
            if c > a2:
                c = a2
            return c * b + (a2 - c) * (k + 1) - (a2 * (a2 + 1) // 2 - c * (c + 1) // 2)

        total = 0
        for sign in (1, -1):
            arr = [sign * x for x in nums]
            left = [0] * n
            st = []
            for i in range(n):
                while st and arr[st[-1]] < arr[i]:
                    st.pop()
                left[i] = i - (st[-1] if st else -1)
                st.append(i)
            st = []
            for i in range(n - 1, -1, -1):
                while st and arr[st[-1]] <= arr[i]:
                    st.pop()
                right = (st[-1] if st else n) - i
                st.append(i)
                total += sign * arr[i] * count(left[i], right)
        return total

for nums, k, exp in examples:
    got = Solution().minMaxSubarraySum(nums, k)
    print(got, 'OK' if got == exp else 'SAI')

20 OK
-6 OK


## 10. Vì sao thuật toán đúng

1. **Mỗi mảng con được gán cho đúng một phần tử làm max:** trong code, `left` dừng tại phần tử `≥ arr[i]` gần nhất bên trái (nên phần bên trái của mảng con chỉ chứa phần tử **nhỏ hơn hẳn**), `right` dừng tại phần tử `> arr[i]` gần nhất bên phải (phần bên phải chứa phần tử **≤**). Vậy `i` sở hữu đúng những mảng con mà nó là lần xuất hiện **trái nhất** của giá trị lớn nhất — duy nhất cho mỗi mảng con.
2. **Tập mảng con mà `i` sở hữu** = `{[l, r] : l ∈ (i − a, i], r ∈ [i, i + b)}`; thêm ràng buộc độ dài ⇒ đếm cặp `(x, y)` như mục 3.3.
3. **Công thức đếm** được chứng minh bằng tách khúc (cách 3) hoặc bao hàm – loại trừ (cách 4), và kiểm chứng bằng máy.
4. **Min** = `−max` của mảng đổi dấu.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Mọi mảng con ≤ k | `O(n k)` | `O(1)` |
| 2. Đóng góp + vòng lặp | `O(n + Σ min(a_i, k))` (tệ nhất `O(n k)`) | `O(n)` |
| 3. Đóng góp + công thức đóng | `O(n)` | `O(n)` |
| 4. Bao hàm – loại trừ | `O(n)` | `O(n)` |

Kết quả tới `~2 · 10^6 · n²/2 ≈ 6·10^{15}` — Python không tràn số; C++/Java cần `long long`.

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [11]:
impls = {'loop': lambda a, k: minmax_contrib(a, k, cnt_loop), 'closed': minmax_closed,
         'ie': lambda a, k: minmax_contrib(a, k, cnt_ie), 'submit': lambda a, k: Solution().minMaxSubarraySum(a, k)}
edge = [([5], 1, 10), ([2, 2], 2, 12), ([2, 2], 1, 8), ([-1, -1, -1], 3, -12), ([3, 1, 2], 3, 23)]
for nums, k, exp in examples + edge:
    assert minmax_brute(nums, k) == exp, (nums, k, minmax_brute(nums, k))
    for name, f in impls.items():
        assert f(nums, k) == exp, (name, nums, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


### 12.2. Đối chiếu ngẫu nhiên với vét cạn (nhiều giá trị trùng)

In [12]:
import random
rng = random.Random(3430)
cnt = 0
for _ in range(3000):
    n = rng.randint(1, 14)
    nums = [rng.randint(-3, 3) for _ in range(n)]
    k = rng.randint(1, n)
    exp = minmax_brute(nums, k)
    for name, f in impls.items():
        assert f(nums, k) == exp, (name, nums, k)
    cnt += 1
print('Khớp vét cạn trên', cnt, 'bộ ngẫu nhiên.')

Khớp vét cạn trên 3000 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = 8·10^4`

In [13]:
import time
n = 8 * 10 ** 4
cases = {
    'ngẫu nhiên, k = n/2': ([rng.randint(-10 ** 6, 10 ** 6) for _ in range(n)], n // 2),
    'tăng dần, k = n': (list(range(n)), n),
    'toàn bằng nhau, k = 1000': ([7] * n, 1000),
}
for name, (nums, k) in cases.items():
    t = time.time(); a = Solution().minMaxSubarraySum(nums, k); t1 = time.time() - t
    t = time.time(); b = minmax_contrib(nums, k, cnt_ie); t2 = time.time() - t
    assert a == b
    print(f"{name:26s}: {a:>22d} | công thức đóng {t1:.2f}s | bao hàm-loại trừ {t2:.2f}s")

nums, k = cases['tăng dần, k = n']
for m in (2500, 5000):
    t = time.time(); c = minmax_contrib(nums[:m], m, cnt_loop); t3 = time.time() - t
    print(f"cách 2 (vòng lặp) trên mảng tăng dần n = k = {m}: {t3:.2f}s — gấp đôi n thì ~4 lần thời gian (O(n·k))")

ngẫu nhiên, k = n/2       :           149093964678 | công thức đóng 0.07s | bao hàm-loại trừ 0.09s


tăng dần, k = n           :        255999999960000 | công thức đóng 0.06s | bao hàm-loại trừ 0.09s


toàn bằng nhau, k = 1000  :             1113007000 | công thức đóng 0.05s | bao hàm-loại trừ 0.07s


cách 2 (vòng lặp) trên mảng tăng dần n = k = 2500: 0.17s — gấp đôi n thì ~4 lần thời gian (O(n·k))


cách 2 (vòng lặp) trên mảng tăng dần n = k = 5000: 0.73s — gấp đôi n thì ~4 lần thời gian (O(n·k))


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Quên giới hạn độ dài (đếm `a · b`)

In [14]:
def wrong_no_k(nums, k):
    return minmax_contrib(nums, k, lambda a, b, kk: a * b)

print('sai :', wrong_no_k([1, 2, 3], 2), '| đúng:', Solution().minMaxSubarraySum([1, 2, 3], 2))

sai : 24 | đúng: 20


**Vì sao sai:** chỉ tính mảng con có **tối đa `k`** phần tử.

### Lỗi 2 — Phá hòa sai: cả hai phía cùng dùng `<` (đếm trùng khi có giá trị bằng nhau)

In [15]:
def wrong_ties(nums, k):
    total = 0
    for sign in (1, -1):
        arr = [sign * x for x in nums]
        n = len(arr)
        left, st = [0] * n, []
        for i in range(n):
            while st and arr[st[-1]] <= arr[i]:          # SAI: hai phía cùng "vượt qua" phần tử bằng
                st.pop()
            left[i] = i - (st[-1] if st else -1); st.append(i)
        right, st = [0] * n, []
        for i in range(n - 1, -1, -1):
            while st and arr[st[-1]] <= arr[i]:
                st.pop()
            right[i] = (st[-1] if st else n) - i; st.append(i)
        total += sign * sum(arr[i] * cnt_closed(left[i], right[i], k) for i in range(n))
    return total

print('sai :', wrong_ties([2, 2], 2), '| đúng:', Solution().minMaxSubarraySum([2, 2], 2))

sai : 16 | đúng: 12


**Vì sao sai:** mảng con `[2, 2]` được tính max **hai lần** (cả hai số 2 đều "sở hữu" nó).

### Lỗi 3 — Đếm `min(a, k) · min(b, k)`

In [16]:
def wrong_product_clamp(nums, k):
    return minmax_contrib(nums, k, lambda a, b, kk: min(a, kk) * min(b, kk))

rng = random.Random(3)
ce = next((a, k) for a, k in (([rng.randint(-3, 3) for _ in range(rng.randint(1, 6))], rng.randint(1, 4)) for _ in range(10000))
          if k <= len(a) and wrong_product_clamp(a, k) != minmax_brute(a, k))
print('phản ví dụ:', ce, '| sai:', wrong_product_clamp(*ce), '| đúng:', Solution().minMaxSubarraySum(*ce))

phản ví dụ: ([-3, 3, 0, -1, 1], 2) | sai: 4 | đúng: 2


**Vì sao sai:** ràng buộc là trên **tổng** `x + y − 1 ≤ k`, không phải trên từng phía.

### Lỗi 4 — Lệch một: dùng `x + y ≤ k`

In [17]:
def wrong_off_by_one(nums, k):
    return minmax_contrib(nums, k, lambda a, b, kk: cnt_closed(a, b, kk - 1) if kk > 1 else 0)

print('sai :', wrong_off_by_one([1, 2, 3], 2), '| đúng:', Solution().minMaxSubarraySum([1, 2, 3], 2))

sai : 12 | đúng: 20


**Vì sao sai:** độ dài mảng con là `x + y − 1` (phần tử `i` được tính ở cả `x` lẫn `y`).

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Tổng max/min trên mọi mảng con" ⇒ đóng góp + ngăn xếp đơn điệu (LC 907, LC 2104).
- Thêm "độ dài ≤ k" ⇒ đếm cặp `(x, y)` dưới đường chéo `x + y ≤ k + 1` — công thức đóng.
- Phá hòa: một phía `<`, phía kia `≤`.
- Min = max của mảng đổi dấu.

## 15. Biến thể của bài

1. **LC 907 — Sum of Subarray Minimums:** không giới hạn độ dài (`k = n`), mod `10^9 + 7`.
2. **LC 2104 — Sum of Subarray Ranges:** `Σ (max − min)`.
3. Độ dài **đúng bằng** `k` ⇒ `cnt_exact = cnt(k) − cnt(k − 1)`.
4. Độ dài trong đoạn `[lo, hi]` ⇒ `cnt(hi) − cnt(lo − 1)`.

In [18]:
# Biến thể 3, 4: độ dài trong [lo, hi]
def minmax_len_range(nums, lo, hi):
    def cnt(a, b, kk):
        return cnt_closed(a, b, hi) - (cnt_closed(a, b, lo - 1) if lo > 1 else 0)
    return minmax_contrib(nums, hi, cnt)

def brute_range(nums, lo, hi):
    n = len(nums)
    return sum(max(nums[i:j]) + min(nums[i:j]) for i in range(n) for j in range(i + 1, n + 1) if lo <= j - i <= hi)

rng = random.Random(15)
for _ in range(500):
    n = rng.randint(1, 10); nums = [rng.randint(-4, 4) for _ in range(n)]
    lo = rng.randint(1, n); hi = rng.randint(lo, n)
    assert minmax_len_range(nums, lo, hi) == brute_range(nums, lo, hi)
print('Biến thể độ dài trong [lo, hi] khớp vét cạn. LC 2104 ví dụ [1,2,3] (max - min) =',
      sum(max([1, 2, 3][i:j]) - min([1, 2, 3][i:j]) for i in range(3) for j in range(i + 1, 4)))

Biến thể độ dài trong [lo, hi] khớp vét cạn. LC 2104 ví dụ [1,2,3] (max - min) = 4


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Mọi mảng con ≤ k | `O(nk)` | Oracle |
| 2 | Đóng góp + vòng lặp | tệ nhất `O(nk)` | Trung gian |
| 3 | Đóng góp + công thức đóng | `O(n)` | **Nộp bài** |
| 4 | Bao hàm – loại trừ | `O(n)` | Công thức gọn hơn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [4]`, `k = 1`.

**Bài 2 (tính tay):** `cnt(a = 3, b = 3, k = 3)`.

**Bài 3 (điền TODO):** viết `sum_subarray_mins(arr)` (LC 907, mod `10^9 + 7`).

**Bài 4 (tự viết lại):** viết `sum_ranges_at_most_k(nums, k)` = `Σ (max − min)` trên mảng con độ dài ≤ k.

**Bài 5 (mini challenge):** với nhiều truy vấn `k` khác nhau trên cùng mảng, trả lời mỗi truy vấn trong `O(log n)` sau tiền xử lý.

In [19]:
def sum_subarray_mins_todo(arr):
    # TODO
    raise NotImplementedError

def sum_ranges_at_most_k_todo(nums, k):
    # TODO
    raise NotImplementedError

def answer_queries_todo(nums, ks):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Một mảng con. (2) max = min = 4. (3) `8`.

**Bài 2.** (1) Cặp `x, y ∈ [1, 3]`, `x + y ≤ 4`. (2) `x = 1`: 3; `x = 2`: 2; `x = 3`: 1. (3) `6`.

**Bài 3.** (1) Đóng góp với `a · b`. (2) Phá hòa. (3) Lấy mod cuối.

**Bài 4.** (1) Max − min = max của `nums` + max của `−nums`. (2) Dùng `minmax_contrib` nhưng không đổi dấu phần min. (3) Đối chiếu vét cạn.

**Bài 5.** (1) `cnt(a, b, k)` tuyến tính từng khúc theo `k` với mốc `min(a,b)`, `max(a,b)`, `a + b − 1`. (2) Tổng các hàm như vậy là đa thức từng khúc ⇒ mảng hiệu bậc 2 theo `k`. (3) Hoặc đơn giản: tính lại `O(n)` mỗi truy vấn (đủ cho ít truy vấn).

## 19. Đáp án tham khảo cho bài tập

In [20]:
# Bài 1, 2, 3
assert Solution().minMaxSubarraySum([4], 1) == 8
assert cnt_closed(3, 3, 3) == 6 == count_pairs_brute(3, 3, 3)
print('Bài 1: 8 | Bài 2:', cnt_closed(3, 3, 3))

def sum_subarray_mins(arr):
    MOD = 10 ** 9 + 7
    neg = [-x for x in arr]
    left, right = extents(neg)
    return sum(arr[i] * left[i] * right[i] for i in range(len(arr))) % MOD

assert sum_subarray_mins([3, 1, 2, 4]) == 17 and sum_subarray_mins([11, 81, 94, 43, 3]) == 444     # ví dụ LC 907
print('Bài 3: LC 907 ví dụ ->', sum_subarray_mins([3, 1, 2, 4]), sum_subarray_mins([11, 81, 94, 43, 3]))

Bài 1: 8 | Bài 2: 6
Bài 3: LC 907 ví dụ -> 17 444


In [21]:
# Bài 4, 5
def sum_ranges_at_most_k(nums, k):
    total = 0
    for sign in (1, -1):
        arr = [sign * x for x in nums]
        left, right = extents(arr)
        total += sum(arr[i] * cnt_closed(left[i], right[i], k) for i in range(len(arr)))
    return total                                   # = Σ max + Σ (−min)

def brute_ranges(nums, k):
    n = len(nums)
    return sum(max(nums[i:j]) - min(nums[i:j]) for i in range(n) for j in range(i + 1, min(n, i + k) + 1))

def answer_queries(nums, ks):
    # tiền xử lý phạm vi một lần, mỗi truy vấn O(n) với công thức đóng
    pre = []
    for sign in (1, -1):
        arr = [sign * x for x in nums]
        left, right = extents(arr)
        pre.append((sign, arr, left, right))
    out = []
    for k in ks:
        out.append(sum(sign * sum(arr[i] * cnt_closed(left[i], right[i], k) for i in range(len(arr)))
                       for sign, arr, left, right in pre))
    return out

rng = random.Random(19)
for _ in range(400):
    n = rng.randint(1, 10); nums = [rng.randint(-5, 5) for _ in range(n)]; k = rng.randint(1, n)
    assert sum_ranges_at_most_k(nums, k) == brute_ranges(nums, k)
nums = [rng.randint(-9, 9) for _ in range(12)]
assert answer_queries(nums, list(range(1, 13))) == [minmax_brute(nums, k) for k in range(1, 13)]
print('Bài 4, 5 khớp vét cạn; LC 2104 [1,3,3] (k = n):', sum_ranges_at_most_k([1, 3, 3], 3))

Bài 4, 5 khớp vét cạn; LC 2104 [1,3,3] (k = n): 4


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
left[i]  = i - (chỉ số gần nhất bên trái có arr >= arr[i])     # dừng ở >=
right[i] = (chỉ số gần nhất bên phải có arr > arr[i]) - i       # dừng ở >
cnt(a, b, k): a' = min(a, k); c = clamp(k+1-b, 0, a')
              = c*b + (a'-c)*(k+1) - (a'(a'+1)/2 - c(c+1)/2)
tổng = Σ arr[i]*cnt   (làm cho arr = nums và arr = -nums, nhân dấu)
```

**Checklist:**

- [ ] Phá hòa một phía `<`, một phía `≤`?
- [ ] Ràng buộc độ dài `x + y ≤ k + 1`?
- [ ] Phần min dùng mảng đổi dấu và nhân lại dấu?
- [ ] Không vòng lặp `O(k)` cho mỗi phần tử?
- [ ] Số lớn (C++/Java: `long long`)?